# Собственная ортогональная модель

Цель — построить нашу модель без чужих OOF-прогнозов, а затем проверить, добавляет ли она новый сигнал к `stack247`.

План эксперимента:

1. Проверить данные, фолды и базовый OOF `stack247`.
2. Создать cross-fitted auxiliary-признаки без утечки таргета.
3. Обучить первую модель на тех же фолдах.
4. Сравнить её standalone AUC и прирост в ансамбле со `stack247`.
5. Только после честной OOF-проверки построить test-прогноз.

На первом шаге ничего не обучаем: убеждаемся, что все строки правильно выровнены.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

## Загружаем одну систему координат

Для OOF-ансамбля особенно важен порядок строк. Если хотя бы один массив сдвинут, AUC станет бессмысленным, хотя код продолжит работать. Поэтому отдельно загружаем сырые данные, сохранённые фолды и базовые прогнозы, а затем связываем их проверками через `assert`.

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
oof_dir = project_root / "artifacts" / "oof"
test_predictions_dir = project_root / "artifacts" / "test_predictions"

train = pd.read_csv(raw_data_dir / "train.csv")
test = pd.read_csv(raw_data_dir / "test.csv")
sample_submission = pd.read_csv(raw_data_dir / "sample_submission.csv")
fold_frame = pd.read_parquet(
    processed_data_dir / "train_with_folds.parquet",
    columns=["id", "fold"],
)

target = train["satisfaction"].astype(np.int8).to_numpy()
fold_numbers = fold_frame["fold"].to_numpy()
stack247_oof = np.load(oof_dir / "stack247.npy").reshape(-1)
stack247_test = np.load(test_predictions_dir / "stack247.npy").reshape(-1)

## Проверяем выравнивание

`assert` останавливает выполнение, если условие неверно. Здесь это полезнее обычного `print`: при ошибке мы не сможем случайно перейти к обучению на перемешанных строках.

После проверок повторно считаем базовый ROC AUC. Ожидаем значение около `0.962205`. Это наша контрольная точка для всех следующих экспериментов.

In [3]:
assert train["id"].equals(fold_frame["id"]), "Фолды не совпадают с порядком train"
assert test["id"].equals(sample_submission["id"]), "sample_submission не совпадает с test"
assert len(stack247_oof) == len(train), "Неверная длина stack247 OOF"
assert len(stack247_test) == len(test), "Неверная длина stack247 test"
assert np.isfinite(stack247_oof).all(), "В stack247 OOF есть NaN или infinity"
assert np.isfinite(stack247_test).all(), "В stack247 test есть NaN или infinity"

stack247_auc = roc_auc_score(target, stack247_oof)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("fold sizes:", pd.Series(fold_numbers).value_counts().sort_index().to_dict())
print(f"stack247 OOF AUC: {stack247_auc:.9f}")

train shape: (699635, 23)
test shape: (299844, 22)
fold sizes: {0: 139927, 1: 139927, 2: 139927, 3: 139927, 4: 139927}
stack247 OOF AUC: 0.962205015


## Следующий шаг

Обычный target encoding мы уже проверяли в предыдущих экспериментах, и рядом со `stack247` он не помог. Поэтому строим другой тип признака: ожидаемое значение одного исходного столбца по контексту остальных столбцов. Целевая переменная `satisfaction` здесь вообще не используется.

## Первый auxiliary-признак: ожидаемый Online boarding

Попробуем оценить, какую оценку `Online boarding` обычно ставят пассажиры с похожим контекстом: тем же классом, типом поездки и похожими оценками других услуг.

Затем создадим два признака:

- `Expected Online boarding` — ожидаемая оценка для такого контекста;
- `Online boarding residual` — разница между фактической и ожидаемой оценкой.

Например, значение residual `-2` означает, что пассажир оценил онлайн-посадку на два балла хуже, чем обычно оценивают похожие пассажиры. Это взаимодействие нескольких столбцов, которое дереву не всегда легко восстановить напрямую.

Для каждой validation-части таблица средних строится только по остальным четырём фолдам. Строка не участвует в вычислении собственного ожидаемого значения.

In [4]:
def make_cross_fitted_expected_value(
    train_frame,
    test_frame,
    value_column,
    group_columns,
    folds,
    smoothing=25.0,
):
    oof_expected = np.full(len(train_frame), np.nan, dtype=np.float32)
    test_expected_by_fold = []

    for valid_fold in np.sort(np.unique(folds)):
        train_mask = folds != valid_fold
        valid_mask = folds == valid_fold

        auxiliary_train = train_frame.loc[
            train_mask,
            group_columns + [value_column],
        ]
        global_mean = auxiliary_train[value_column].mean()

        group_stats = (
            auxiliary_train
            .groupby(group_columns, dropna=False)[value_column]
            .agg(["sum", "count"])
        )
        group_stats["expected"] = (
            group_stats["sum"] + smoothing * global_mean
        ) / (group_stats["count"] + smoothing)
        mapping = group_stats["expected"]

        valid_keys = pd.MultiIndex.from_frame(
            train_frame.loc[valid_mask, group_columns]
        )
        valid_expected = mapping.reindex(valid_keys).to_numpy()
        oof_expected[valid_mask] = np.nan_to_num(
            valid_expected,
            nan=global_mean,
        )

        test_keys = pd.MultiIndex.from_frame(test_frame[group_columns])
        fold_test_expected = mapping.reindex(test_keys).to_numpy()
        fold_test_expected = np.nan_to_num(
            fold_test_expected,
            nan=global_mean,
        )
        test_expected_by_fold.append(fold_test_expected)

    test_expected = np.mean(test_expected_by_fold, axis=0).astype(np.float32)
    return oof_expected, test_expected

In [5]:
auxiliary_group_columns = [
    "Class",
    "Type of Travel",
    "Customer Type",
    "Inflight wifi service",
    "Seat comfort",
    "Inflight entertainment",
]

expected_online_oof, expected_online_test = make_cross_fitted_expected_value(
    train_frame=train,
    test_frame=test,
    value_column="Online boarding",
    group_columns=auxiliary_group_columns,
    folds=fold_numbers,
    smoothing=25.0,
)

train_auxiliary = pd.DataFrame({
    "Expected Online boarding": expected_online_oof,
    "Online boarding residual": (
        train["Online boarding"].to_numpy() - expected_online_oof
    ),
})
test_auxiliary = pd.DataFrame({
    "Expected Online boarding": expected_online_test,
    "Online boarding residual": (
        test["Online boarding"].to_numpy() - expected_online_test
    ),
})

assert np.isfinite(train_auxiliary.to_numpy()).all()
assert np.isfinite(test_auxiliary.to_numpy()).all()

observed_expected_correlation = np.corrcoef(
    train["Online boarding"].to_numpy(),
    expected_online_oof,
)[0, 1]

print("train auxiliary shape:", train_auxiliary.shape)
print("test auxiliary shape:", test_auxiliary.shape)
print(f"observed/expected correlation: {observed_expected_correlation:.6f}")
print(train_auxiliary.describe().round(4))

train auxiliary shape: (699635, 2)
test auxiliary shape: (299844, 2)
observed/expected correlation: 0.860228
       Expected Online boarding  Online boarding residual
count               699635.0000               699635.0000
mean                     3.3773                   -0.0132
std                      1.0752                    0.6579
min                      0.3170                   -4.8805
25%                      2.7302                   -0.2993
50%                      3.5970                   -0.0340
75%                      4.3752                    0.4271
max                      4.8844                    4.6830


## Что будем делать после проверки

Пока мы не знаем, полезны ли эти два столбца для ROC AUC. Следующий честный тест — добавить их к одной фиксированной LightGBM-модели и сравнить baseline и auxiliary-вариант на одинаковых фолдах и параметрах. Одновременно менять параметры модели нельзя: иначе мы не поймём, что именно дало разницу.

## A/B-тест на одном фолде

Сначала используем только фолд 0, чтобы быстро проверить направление.

- **Baseline** получает только исходные очищенные признаки.
- **Auxiliary** получает те же признаки и два новых столбца.

У моделей одинаковые параметры, обучающие строки и validation-строки. Поэтому разницу AUC можно связать именно с новыми признаками. Это пока диагностический эксперимент, а не финальная модель и не submission.

In [6]:
import gc

import lightgbm as lgb

model_train = pd.read_parquet(processed_data_dir / "train_with_folds.parquet")
model_test = pd.read_parquet(processed_data_dir / "test_clean.parquet")

assert model_train["id"].equals(train["id"])
assert model_test["id"].equals(test["id"])

model_feature_columns = [
    column
    for column in model_train.columns
    if column not in ["id", "satisfaction", "fold"]
]

ablation_parameters = {
    "n_estimators": 3000,
    "learning_rate": 0.03,
    "num_leaves": 127,
    "min_child_samples": 50,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.3,
    "reg_lambda": 1.0,
    "n_jobs": 4,
    "verbosity": -1,
}

print("base feature count:", len(model_feature_columns))
print("auxiliary feature count:", train_auxiliary.shape[1])

base feature count: 22
auxiliary feature count: 2


In [7]:
def train_one_ablation_model(valid_fold, use_auxiliary):
    train_mask = model_train["fold"].to_numpy() != valid_fold
    valid_mask = model_train["fold"].to_numpy() == valid_fold

    x_train = model_train.loc[train_mask, model_feature_columns].copy()
    x_valid = model_train.loc[valid_mask, model_feature_columns].copy()
    y_train = model_train.loc[train_mask, "satisfaction"]
    y_valid = model_train.loc[valid_mask, "satisfaction"]

    if use_auxiliary:
        x_train = pd.concat([
            x_train,
            train_auxiliary.loc[train_mask],
        ], axis=1)
        x_valid = pd.concat([
            x_valid,
            train_auxiliary.loc[valid_mask],
        ], axis=1)

    arrival_column = "Arrival Delay in Minutes"
    arrival_median = x_train[arrival_column].median()
    x_train[arrival_column] = x_train[arrival_column].fillna(arrival_median)
    x_valid[arrival_column] = x_valid[arrival_column].fillna(arrival_median)

    class_categories = sorted(
        set(x_train["Class"].dropna().unique())
        | set(x_valid["Class"].dropna().unique())
    )
    for frame in [x_train, x_valid]:
        frame["Class"] = pd.Categorical(
            frame["Class"],
            categories=class_categories,
        )

    model = lgb.LGBMClassifier(
        **ablation_parameters,
        random_state=42 + valid_fold,
    )
    model.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(150),
            lgb.log_evaluation(100),
        ],
    )

    valid_prediction = model.predict_proba(
        x_valid,
        num_iteration=model.best_iteration_,
    )[:, 1]
    fold_auc = roc_auc_score(y_valid, valid_prediction)
    best_iteration = model.best_iteration_

    del model, x_train, x_valid, y_train, y_valid
    gc.collect()

    return fold_auc, best_iteration, valid_prediction

In [8]:
fold_to_test = 0

baseline_auc, baseline_iteration, baseline_prediction = (
    train_one_ablation_model(
        valid_fold=fold_to_test,
        use_auxiliary=False,
    )
)

auxiliary_auc, auxiliary_iteration, auxiliary_prediction = (
    train_one_ablation_model(
        valid_fold=fold_to_test,
        use_auxiliary=True,
    )
)

auc_difference = auxiliary_auc - baseline_auc
prediction_correlation = np.corrcoef(
    baseline_prediction,
    auxiliary_prediction,
)[0, 1]

print(f"baseline AUC:  {baseline_auc:.9f}")
print(f"auxiliary AUC: {auxiliary_auc:.9f}")
print(f"difference:    {auc_difference:+.9f}")
print("baseline best iteration:", baseline_iteration)
print("auxiliary best iteration:", auxiliary_iteration)
print(f"prediction correlation: {prediction_correlation:.9f}")

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.956955	valid_0's binary_logloss: 0.237903
[200]	valid_0's auc: 0.958707	valid_0's binary_logloss: 0.227598
[300]	valid_0's auc: 0.959202	valid_0's binary_logloss: 0.226428
[400]	valid_0's auc: 0.959289	valid_0's binary_logloss: 0.22614
[500]	valid_0's auc: 0.959342	valid_0's binary_logloss: 0.226018
[600]	valid_0's auc: 0.959287	valid_0's binary_logloss: 0.226002
Early stopping, best iteration is:
[462]	valid_0's auc: 0.959354	valid_0's binary_logloss: 0.226039


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.956914	valid_0's binary_logloss: 0.237186
[200]	valid_0's auc: 0.958581	valid_0's binary_logloss: 0.228395
[300]	valid_0's auc: 0.959053	valid_0's binary_logloss: 0.227263
[400]	valid_0's auc: 0.959144	valid_0's binary_logloss: 0.226941
[500]	valid_0's auc: 0.959203	valid_0's binary_logloss: 0.226727
[600]	valid_0's auc: 0.959205	valid_0's binary_logloss: 0.226614
Early stopping, best iteration is:
[508]	valid_0's auc: 0.95921	valid_0's binary_logloss: 0.226714
baseline AUC:  0.959353668
auxiliary AUC: 0.959210245
difference:    -0.000143423
baseline best iteration: 462
auxiliary best iteration: 508
prediction correlation: 0.998365785


## Как читать результат

Положительный `difference` означает, что на фолде 0 auxiliary-вариант лучше baseline. Но одного фолда недостаточно для решения: небольшой плюс может оказаться шумом. Если результат положительный, повторим A/B-тест на всех пяти фолдах и посмотрим не только средний прирост, но и его направление на каждом фолде.

## Учимся исправлять stack247

Предыдущий эксперимент показал, что два ручных признака не улучшают обычную модель. Теперь используем более прямой подход: маленькая LightGBM начнёт обучение не с нулевого прогноза, а с уже сильного прогноза `stack247`.

Деревья будут учить только поправку к базовому прогнозу. Если `stack247` уже прав для некоторого типа пассажиров, большая поправка не нужна. Если в его ошибках есть повторяющийся рисунок, модель сможет его найти по исходным признакам.

Работаем в logit-шкале, потому что именно в ней бинарная модель складывает базовый прогноз и поправку:

`final logit = stack247 logit + alpha × correction`

`alpha = 0` означает не менять `stack247`; `alpha = 1` означает применить всю поправку. Пока проверяем только фолд 0.

In [9]:
def probability_to_logit(probability, epsilon=1e-6):
    clipped = np.clip(probability, epsilon, 1.0 - epsilon)
    return np.log(clipped / (1.0 - clipped))


def train_stack_residual_corrector(valid_fold):
    train_mask = model_train["fold"].to_numpy() != valid_fold
    valid_mask = model_train["fold"].to_numpy() == valid_fold

    x_train = model_train.loc[train_mask, model_feature_columns].copy()
    x_valid = model_train.loc[valid_mask, model_feature_columns].copy()
    y_train = model_train.loc[train_mask, "satisfaction"].to_numpy()
    y_valid = model_train.loc[valid_mask, "satisfaction"].to_numpy()

    base_train_logit = probability_to_logit(stack247_oof[train_mask])
    base_valid_logit = probability_to_logit(stack247_oof[valid_mask])

    arrival_column = "Arrival Delay in Minutes"
    arrival_median = x_train[arrival_column].median()
    x_train[arrival_column] = x_train[arrival_column].fillna(arrival_median)
    x_valid[arrival_column] = x_valid[arrival_column].fillna(arrival_median)

    class_categories = sorted(
        set(x_train["Class"].dropna().unique())
        | set(x_valid["Class"].dropna().unique())
    )
    for frame in [x_train, x_valid]:
        frame["Class"] = pd.Categorical(
            frame["Class"],
            categories=class_categories,
        )

    residual_parameters = {
        "n_estimators": 1200,
        "learning_rate": 0.02,
        "num_leaves": 31,
        "max_depth": 6,
        "min_child_samples": 500,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.8,
        "reg_alpha": 1.0,
        "reg_lambda": 5.0,
        "n_jobs": 4,
        "verbosity": -1,
    }

    corrector = lgb.LGBMClassifier(
        **residual_parameters,
        random_state=1000 + valid_fold,
    )
    corrector.fit(
        x_train,
        y_train,
        init_score=base_train_logit,
        eval_set=[(x_valid, y_valid)],
        eval_init_score=[base_valid_logit],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(50),
        ],
    )

    correction = corrector.predict(
        x_valid,
        raw_score=True,
        num_iteration=corrector.best_iteration_,
    )
    best_iteration = corrector.best_iteration_

    del corrector, x_train, x_valid, y_train
    gc.collect()

    return y_valid, base_valid_logit, correction, best_iteration

In [10]:
residual_fold = 0

(
    residual_y_valid,
    residual_base_logit,
    residual_correction,
    residual_best_iteration,
) = train_stack_residual_corrector(residual_fold)

correction_strengths = [0.0, 0.10, 0.25, 0.50, 0.75, 1.00]
residual_scores = {}

for strength in correction_strengths:
    corrected_logit = (
        residual_base_logit + strength * residual_correction
    )
    residual_scores[strength] = roc_auc_score(
        residual_y_valid,
        corrected_logit,
    )

base_residual_auc = residual_scores[0.0]
best_strength = max(residual_scores, key=residual_scores.get)
best_residual_auc = residual_scores[best_strength]

print("best iteration:", residual_best_iteration)
print(f"correction mean: {residual_correction.mean():+.6f}")
print(f"correction std:  {residual_correction.std():.6f}")
print()
for strength, score in residual_scores.items():
    print(
        f"alpha={strength:>4.2f}  "
        f"AUC={score:.9f}  "
        f"delta={score - base_residual_auc:+.9f}"
    )

print()
print(f"best alpha: {best_strength:.2f}")
print(f"best delta: {best_residual_auc - base_residual_auc:+.9f}")

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[50]	valid_0's auc: 0.962332	valid_0's binary_logloss: 0.219502
[100]	valid_0's auc: 0.962306	valid_0's binary_logloss: 0.219534
Early stopping, best iteration is:
[1]	valid_0's auc: 0.962357	valid_0's binary_logloss: 0.219468
best iteration: 1
correction mean: +0.000044
correction std:  0.001296

alpha=0.00  AUC=0.962357578  delta=+0.000000000
alpha=0.10  AUC=0.962357579  delta=+0.000000000
alpha=0.25  AUC=0.962357525  delta=-0.000000053
alpha=0.50  AUC=0.962357417  delta=-0.000000162
alpha=0.75  AUC=0.962357393  delta=-0.000000185
alpha=1.00  AUC=0.962357231  delta=-0.000000347

best alpha: 0.10
best delta: +0.000000000


## Как примем решение

Если лучший `alpha` равен нулю, модель не нашла полезной поправки. Если небольшой положительный `alpha` улучшает фолд 0, это только предварительный сигнал: затем нужно построить отдельную correction-модель для каждого из пяти фолдов и проверить устойчивость прироста. Вес, выбранный по одному фолду, в финальный submission не переносим.

## Новый независимый сигнал: разряды чисел

Перестраивать корректор дальше не будем: на фолде 0 он остановился после первого дерева и не дал измеримого прироста. Вместо этого проверим новый блок признаков.

Для `Age`, `Flight Distance` и двух задержек выделим единицы, десятки, сотни и тысячи. Например, из расстояния `1847` получим четыре дополнительных значения: `7`, `4`, `8`, `1`.

Такие признаки не используют `satisfaction` и не создают утечку. Они могут помочь на синтетических данных, если генератор оставил закономерность в отдельных разрядах. Но это только гипотеза, поэтому снова проводим A/B-тест на фолде 0.

In [11]:
digit_source_columns = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
]
digit_places = [1, 10, 100, 1000]


def add_digit_features(frame):
    for column in digit_source_columns:
        integer_values = np.floor(
            np.abs(frame[column].to_numpy())
        ).astype(np.int64)

        for place in digit_places:
            feature_name = f"{column} digit {place}"
            frame[feature_name] = (integer_values // place) % 10

    return frame

In [12]:
def train_digit_ablation_model(valid_fold):
    train_mask = model_train["fold"].to_numpy() != valid_fold
    valid_mask = model_train["fold"].to_numpy() == valid_fold

    x_train = model_train.loc[train_mask, model_feature_columns].copy()
    x_valid = model_train.loc[valid_mask, model_feature_columns].copy()
    y_train = model_train.loc[train_mask, "satisfaction"]
    y_valid = model_train.loc[valid_mask, "satisfaction"]

    arrival_column = "Arrival Delay in Minutes"
    arrival_median = x_train[arrival_column].median()
    x_train[arrival_column] = x_train[arrival_column].fillna(arrival_median)
    x_valid[arrival_column] = x_valid[arrival_column].fillna(arrival_median)

    x_train = add_digit_features(x_train)
    x_valid = add_digit_features(x_valid)

    class_categories = sorted(
        set(x_train["Class"].dropna().unique())
        | set(x_valid["Class"].dropna().unique())
    )
    for frame in [x_train, x_valid]:
        frame["Class"] = pd.Categorical(
            frame["Class"],
            categories=class_categories,
        )

    digit_model = lgb.LGBMClassifier(
        **ablation_parameters,
        random_state=42 + valid_fold,
    )
    digit_model.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(150),
            lgb.log_evaluation(100),
        ],
    )

    digit_prediction = digit_model.predict_proba(
        x_valid,
        num_iteration=digit_model.best_iteration_,
    )[:, 1]
    digit_auc = roc_auc_score(y_valid, digit_prediction)
    digit_best_iteration = digit_model.best_iteration_
    digit_feature_count = x_train.shape[1]

    del digit_model, x_train, x_valid, y_train, y_valid
    gc.collect()

    return (
        digit_auc,
        digit_best_iteration,
        digit_feature_count,
        digit_prediction,
    )

In [13]:
(
    digit_auc,
    digit_best_iteration,
    digit_feature_count,
    digit_prediction,
) = train_digit_ablation_model(valid_fold=0)

digit_difference = digit_auc - baseline_auc
digit_prediction_correlation = np.corrcoef(
    baseline_prediction,
    digit_prediction,
)[0, 1]

print(f"baseline AUC: {baseline_auc:.9f}")
print(f"digit AUC:    {digit_auc:.9f}")
print(f"difference:   {digit_difference:+.9f}")
print("digit best iteration:", digit_best_iteration)
print("feature count with digits:", digit_feature_count)
print(f"prediction correlation: {digit_prediction_correlation:.9f}")

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.957012	valid_0's binary_logloss: 0.237035
[200]	valid_0's auc: 0.95887	valid_0's binary_logloss: 0.227485
[300]	valid_0's auc: 0.959556	valid_0's binary_logloss: 0.226128
[400]	valid_0's auc: 0.95973	valid_0's binary_logloss: 0.225647
[500]	valid_0's auc: 0.959766	valid_0's binary_logloss: 0.22548
[600]	valid_0's auc: 0.959777	valid_0's binary_logloss: 0.22539
[700]	valid_0's auc: 0.959796	valid_0's binary_logloss: 0.225311
[800]	valid_0's auc: 0.959807	valid_0's binary_logloss: 0.22526
Early stopping, best iteration is:
[652]	valid_0's auc: 0.959829	valid_0's binary_logloss: 0.225308
baseline AUC: 0.959353668
digit AUC:    0.959828777
difference:   +0.000475109
digit best iteration: 652
feature count with digits: 38
prediction correlation: 0.999187975


## Как продолжим

Если разряды дадут заметный положительный результат на фолде 0, проверим тот же неизменный блок на остальных фолдах. Если результат нулевой или отрицательный, не будем расширять его дополнительными цифрами и взаимодействиями: это был бы поиск случайного выигрыша.

## Проверяем разряды на всех фолдах

Фолд 0 уже готов. Теперь обучаем baseline и digit-вариант на фолдах 1–4. После каждого фолда сохраняем OOF-массивы, чтобы не потерять прогресс при остановке ядра.

Признак считаем устойчивым, если он улучшает хотя бы четыре фолда из пяти, а средний прирост заметно больше разброса между фолдами. Только среднее значение без проверки отдельных фолдов может скрыть случайный большой выигрыш на одном фолде.

In [14]:
digit_baseline_oof = np.full(len(model_train), np.nan, dtype=np.float32)
digit_model_oof = np.full(len(model_train), np.nan, dtype=np.float32)
digit_fold_results = {}

fold_zero_mask = fold_numbers == 0
digit_baseline_oof[fold_zero_mask] = baseline_prediction.astype(np.float32)
digit_model_oof[fold_zero_mask] = digit_prediction.astype(np.float32)
digit_fold_results[0] = {
    "baseline_auc": float(baseline_auc),
    "digit_auc": float(digit_auc),
    "difference": float(digit_auc - baseline_auc),
    "baseline_iteration": int(baseline_iteration),
    "digit_iteration": int(digit_best_iteration),
}

for valid_fold in [1, 2, 3, 4]:
    print(f"\n===== fold {valid_fold} =====")

    (
        fold_baseline_auc,
        fold_baseline_iteration,
        fold_baseline_prediction,
    ) = train_one_ablation_model(
        valid_fold=valid_fold,
        use_auxiliary=False,
    )

    (
        fold_digit_auc,
        fold_digit_iteration,
        _,
        fold_digit_prediction,
    ) = train_digit_ablation_model(valid_fold=valid_fold)

    valid_mask = fold_numbers == valid_fold
    digit_baseline_oof[valid_mask] = (
        fold_baseline_prediction.astype(np.float32)
    )
    digit_model_oof[valid_mask] = (
        fold_digit_prediction.astype(np.float32)
    )

    fold_difference = fold_digit_auc - fold_baseline_auc
    digit_fold_results[valid_fold] = {
        "baseline_auc": float(fold_baseline_auc),
        "digit_auc": float(fold_digit_auc),
        "difference": float(fold_difference),
        "baseline_iteration": int(fold_baseline_iteration),
        "digit_iteration": int(fold_digit_iteration),
    }

    np.save(
        oof_dir / "digit_baseline_ablation_partial.npy",
        digit_baseline_oof,
    )
    np.save(
        oof_dir / "digit_lightgbm_partial.npy",
        digit_model_oof,
    )

    print(f"baseline AUC: {fold_baseline_auc:.9f}")
    print(f"digit AUC:    {fold_digit_auc:.9f}")
    print(f"difference:   {fold_difference:+.9f}")
    print("saved partial OOF")

    del fold_baseline_prediction, fold_digit_prediction
    gc.collect()

assert np.isfinite(digit_baseline_oof).all()
assert np.isfinite(digit_model_oof).all()

np.save(oof_dir / "digit_baseline_ablation.npy", digit_baseline_oof)
np.save(oof_dir / "digit_lightgbm.npy", digit_model_oof)


===== fold 1 =====


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.955678	valid_0's binary_logloss: 0.239908
[200]	valid_0's auc: 0.957282	valid_0's binary_logloss: 0.23044
[300]	valid_0's auc: 0.957685	valid_0's binary_logloss: 0.22946
[400]	valid_0's auc: 0.957719	valid_0's binary_logloss: 0.229274
[500]	valid_0's auc: 0.957777	valid_0's binary_logloss: 0.229188
[600]	valid_0's auc: 0.957818	valid_0's binary_logloss: 0.229119
[700]	valid_0's auc: 0.957778	valid_0's binary_logloss: 0.229165
Early stopping, best iteration is:
[587]	valid_0's auc: 0.957828	valid_0's binary_logloss: 0.229123


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.955638	valid_0's binary_logloss: 0.239949
[200]	valid_0's auc: 0.95754	valid_0's binary_logloss: 0.230206
[300]	valid_0's auc: 0.958211	valid_0's binary_logloss: 0.228846
[400]	valid_0's auc: 0.958363	valid_0's binary_logloss: 0.228577
[500]	valid_0's auc: 0.958439	valid_0's binary_logloss: 0.228485
[600]	valid_0's auc: 0.958507	valid_0's binary_logloss: 0.22838
[700]	valid_0's auc: 0.958521	valid_0's binary_logloss: 0.228338
[800]	valid_0's auc: 0.958555	valid_0's binary_logloss: 0.228291
Early stopping, best iteration is:
[746]	valid_0's auc: 0.958566	valid_0's binary_logloss: 0.228283
baseline AUC: 0.957828143
digit AUC:    0.958566440
difference:   +0.000738297
saved partial OOF

===== fold 2 =====


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


... (7 outputs left out)

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.956963	valid_0's binary_logloss: 0.237846
[200]	valid_0's auc: 0.958754	valid_0's binary_logloss: 0.227794
[300]	valid_0's auc: 0.95927	valid_0's binary_logloss: 0.226541
[400]	valid_0's auc: 0.959377	valid_0's binary_logloss: 0.226225
[500]	valid_0's auc: 0.959345	valid_0's binary_logloss: 0.226108
Early stopping, best iteration is:
[390]	valid_0's auc: 0.959393	valid_0's binary_logloss: 0.226228


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 150 rounds
[100]	valid_0's auc: 0.957028	valid_0's binary_logloss: 0.237886
[200]	valid_0's auc: 0.959015	valid_0's binary_logloss: 0.227307
[300]	valid_0's auc: 0.959702	valid_0's binary_logloss: 0.225951
[400]	valid_0's auc: 0.959835	valid_0's binary_logloss: 0.225602
[500]	valid_0's auc: 0.959862	valid_0's binary_logloss: 0.225478
[600]	valid_0's auc: 0.959906	valid_0's binary_logloss: 0.225346
[700]	valid_0's auc: 0.959919	valid_0's binary_logloss: 0.225312
[800]	valid_0's auc: 0.959883	valid_0's binary_logloss: 0.225349
Early stopping, best iteration is:
[691]	valid_0's auc: 0.959929	valid_0's binary_logloss: 0.2253
baseline AUC: 0.959393371
digit AUC:    0.959928721
difference:   +0.000535350
saved partial OOF


In [15]:
digit_results_table = (
    pd.DataFrame.from_dict(digit_fold_results, orient="index")
    .rename_axis("fold")
    .sort_index()
)

positive_fold_count = int(
    (digit_results_table["difference"] > 0).sum()
)
mean_difference = digit_results_table["difference"].mean()
difference_std = digit_results_table["difference"].std(ddof=1)
difference_se = difference_std / np.sqrt(len(digit_results_table))
overall_baseline_auc = roc_auc_score(target, digit_baseline_oof)
overall_digit_auc = roc_auc_score(target, digit_model_oof)

display(
    digit_results_table[[
        "baseline_auc",
        "digit_auc",
        "difference",
    ]].style.format("{:.9f}")
)

print(f"positive folds: {positive_fold_count}/5")
print(f"mean fold difference: {mean_difference:+.9f}")
print(f"difference standard error: {difference_se:.9f}")
print(f"overall baseline OOF AUC: {overall_baseline_auc:.9f}")
print(f"overall digit OOF AUC:    {overall_digit_auc:.9f}")
print(f"overall difference:       {overall_digit_auc - overall_baseline_auc:+.9f}")

,baseline_auc,digit_auc,difference
fold,,,
0,0.959353668,0.959828777,0.000475109
1,0.957828143,0.958566440,0.000738297
2,0.960030446,0.960432886,0.000402440
3,0.959074021,0.959691994,0.000617973
4,0.959393371,0.959928721,0.000535350


positive folds: 5/5
mean fold difference: +0.000553834
difference standard error: 0.000058138
overall baseline OOF AUC: 0.959124501
overall digit OOF AUC:    0.959679504
overall difference:       +0.000555003


## Следующее решение

Если выигрыш повторится минимум на четырёх фолдах, следующим шагом проверим ранговое смешивание полного digit OOF со `stack247`. Это отдельный и более строгий тест: улучшение слабой baseline-модели ещё не означает, что новый сигнал отсутствует внутри большого публичного стека.

## Проверяем, добавляют ли разряды новый сигнал к stack247

Digit-модель заметно слабее большого стека сама по себе. Поэтому большой вес ей не нужен: перебираем небольшую долю от 0% до 25%. Проверяем два способа смешивания:

- `probability` — смешиваем исходные вероятности;
- `rank` — сначала заменяем прогнозы их местами в общем рейтинге. Для ROC AUC это часто надёжнее, потому что масштабы моделей перестают мешать.

Для каждого веса считаем общий AUC и изменение на каждом из пяти фолдов.

In [17]:
digit_oof = np.load(oof_dir / "digit_lightgbm.npy").reshape(-1)

assert len(digit_oof) == len(target)
assert np.isfinite(digit_oof).all()

stack_rank = pd.Series(stack247_oof).rank(method="average", pct=True).to_numpy()
digit_rank = pd.Series(digit_oof).rank(method="average", pct=True).to_numpy()

blend_sources = {
    "probability": (stack247_oof, digit_oof),
    "rank": (stack_rank, digit_rank),
}
digit_weight_grid = np.round(np.arange(0.0, 0.251, 0.005), 3)
stack_fold_auc = {
    fold: roc_auc_score(
        target[fold_numbers == fold],
        stack247_oof[fold_numbers == fold],
    )
    for fold in range(5)
}

blend_grid_rows = []

for blend_method, (stack_source, digit_source) in blend_sources.items():
    for digit_weight in digit_weight_grid:
        blended_oof = (
            (1.0 - digit_weight) * stack_source
            + digit_weight * digit_source
        )

        fold_differences = []
        for fold in range(5):
            valid_mask = fold_numbers == fold
            blended_fold_auc = roc_auc_score(
                target[valid_mask],
                blended_oof[valid_mask],
            )
            fold_differences.append(
                blended_fold_auc - stack_fold_auc[fold]
            )

        blend_auc = roc_auc_score(target, blended_oof)
        blend_grid_rows.append({
            "method": blend_method,
            "digit_weight": float(digit_weight),
            "overall_auc": float(blend_auc),
            "overall_difference": float(blend_auc - stack247_auc),
            "mean_fold_difference": float(np.mean(fold_differences)),
            "minimum_fold_difference": float(np.min(fold_differences)),
            "positive_folds": int(np.sum(np.array(fold_differences) > 0)),
        })

blend_grid_results = pd.DataFrame(blend_grid_rows)
best_blend_candidates = (
    blend_grid_results
    .sort_values(
        ["positive_folds", "overall_auc"],
        ascending=[False, False],
    )
    .head(12)
)

print(best_blend_candidates.to_string(index=False, float_format=lambda value: f"{value:.9f}"))

     method  digit_weight  overall_auc  overall_difference  mean_fold_difference  minimum_fold_difference  positive_folds
probability   0.015000000  0.962205933         0.000000918           0.000000911             -0.000004870               3
probability   0.020000000  0.962205893         0.000000879           0.000000913             -0.000006889               3
probability   0.010000000  0.962205781         0.000000767           0.000000781             -0.000003247               3
probability   0.025000000  0.962205673         0.000000658           0.000000677             -0.000008856               3
probability   0.005000000  0.962205443         0.000000428           0.000000432             -0.000001505               3
probability   0.030000000  0.962205313         0.000000298           0.000000326             -0.000011396               3
       rank   0.010000000  0.962205208         0.000000193           0.000000204             -0.000003173               3
       rank   0.00500000

## Более честная проверка веса

Красивый вес можно случайно подобрать под те же данные, на которых мы показываем результат. Поэтому ниже каждый фолд по очереди полностью прячется. Вес выбирается по другим четырём фолдам и только потом проверяется на спрятанном. Это называется leave-one-fold-out и лучше показывает, перенесётся ли улучшение на неизвестные данные.

In [18]:
lofo_rows = []
lofo_predictions = {
    method: np.full(len(target), np.nan, dtype=np.float64)
    for method in blend_sources
}

for blend_method, (stack_source, digit_source) in blend_sources.items():
    for held_out_fold in range(5):
        selection_mask = fold_numbers != held_out_fold
        held_out_mask = fold_numbers == held_out_fold

        selection_scores = []
        for digit_weight in digit_weight_grid:
            selection_blend = (
                (1.0 - digit_weight) * stack_source[selection_mask]
                + digit_weight * digit_source[selection_mask]
            )
            selection_scores.append(
                roc_auc_score(target[selection_mask], selection_blend)
            )

        best_weight_index = int(np.argmax(selection_scores))
        selected_weight = float(digit_weight_grid[best_weight_index])
        held_out_prediction = (
            (1.0 - selected_weight) * stack_source[held_out_mask]
            + selected_weight * digit_source[held_out_mask]
        )
        lofo_predictions[blend_method][held_out_mask] = held_out_prediction

        held_out_auc = roc_auc_score(
            target[held_out_mask],
            held_out_prediction,
        )
        held_out_difference = held_out_auc - stack_fold_auc[held_out_fold]
        lofo_rows.append({
            "method": blend_method,
            "held_out_fold": held_out_fold,
            "selected_digit_weight": selected_weight,
            "held_out_auc": float(held_out_auc),
            "held_out_difference": float(held_out_difference),
        })

lofo_results = pd.DataFrame(lofo_rows)
print(lofo_results.to_string(index=False, float_format=lambda value: f"{value:.9f}"))

print("\nLOFO summary")
for blend_method, method_rows in lofo_results.groupby("method"):
    nested_auc = roc_auc_score(target, lofo_predictions[blend_method])
    positive_folds = int((method_rows["held_out_difference"] > 0).sum())
    print(
        f"{blend_method}: AUC={nested_auc:.9f}, "
        f"difference={nested_auc - stack247_auc:+.9f}, "
        f"positive folds={positive_folds}/5, "
        f"weights={method_rows['selected_digit_weight'].tolist()}"
    )

     method  held_out_fold  selected_digit_weight  held_out_auc  held_out_difference
probability              0            0.010000000   0.962360301          0.000002723
probability              1            0.030000000   0.961400537         -0.000009826
probability              2            0.015000000   0.962806219          0.000002375
probability              3            0.030000000   0.962434674         -0.000011397
probability              4            0.000000000   0.962040513          0.000000000
       rank              0            0.010000000   0.962357599          0.000000021
       rank              1            0.025000000   0.961401600         -0.000008763
       rank              2            0.000000000   0.962803844          0.000000000
       rank              3            0.015000000   0.962442461         -0.000003609
       rank              4            0.000000000   0.962040513          0.000000000

LOFO summary
probability: AUC=0.962202175, difference=-0.0000028

## Как принимаем решение

Продолжаем только в том случае, если LOFO-проверка сохраняет положительный общий результат и улучшает большинство спрятанных фолдов. Если лучший вес постоянно прыгает или выигрыш остаётся только в общей таблице, считаем его подгонкой и не тратим время на обучение test-прогноза.

## Осторожная смесь V54 и stack247

Digit-кандидат отбрасываем: честная проверка показала шум. Следующий быстрый кандидат использует два уже сильных test-прогноза. `V54` имеет лучший public score, поэтому оставляем ему 85% веса, а `stack247` даём только 15%.

Смешиваем ранги, а не исходные вероятности: ROC AUC оценивает порядок пассажиров, а ранги убирают разницу в масштабе и калибровке двух ансамблей. У V54 нет OOF, поэтому это осознанная leaderboard-проверка, а не локально доказанный прирост.

In [19]:
v54_path = project_root / "submissions" / "rabari_v54_submission.csv"
v54_submission = pd.read_csv(v54_path)

assert v54_submission["id"].equals(sample_submission["id"])
assert np.isfinite(v54_submission["satisfaction"]).all()

v54_test = v54_submission["satisfaction"].to_numpy()
v54_test_rank = pd.Series(v54_test).rank(method="average", pct=True).to_numpy()
stack247_test_rank = pd.Series(stack247_test).rank(method="average", pct=True).to_numpy()

v54_weight = 0.85
stack247_weight = 0.15
v54_stack15_prediction = (
    v54_weight * v54_test_rank
    + stack247_weight * stack247_test_rank
)

v54_stack15_submission = sample_submission.copy()
v54_stack15_submission["satisfaction"] = v54_stack15_prediction

v54_stack15_path = (
    project_root / "submissions" / "blend_v54_85_stack247_15.csv"
)
v54_stack15_submission.to_csv(v54_stack15_path, index=False)

rank_correlation = np.corrcoef(v54_test_rank, stack247_test_rank)[0, 1]
print(f"rank correlation: {rank_correlation:.9f}")
print(f"rows: {len(v54_stack15_submission):,}")
print(f"prediction range: {v54_stack15_prediction.min():.6f} .. {v54_stack15_prediction.max():.6f}")
print(f"saved: {v54_stack15_path}")

rank correlation: 0.996548757
rows: 299,844
prediction range: 0.000010 .. 0.999999
saved: c:\Users\User\Desktop\Airline Satisfaction\submissions\blend_v54_85_stack247_15.csv


Загружаем только файл `blend_v54_85_stack247_15.csv`. Если он не обгонит чистый V54, не подбираем вес по десяткам public-попыток: это быстро превращается в переобучение на leaderboard.

## Pairwise-переранжировщик поверх stack247

Обычные модели минимизируют ошибку вероятности. Здесь задача другая: `XGBRanker` получает пары пассажиров и учится ставить удовлетворённого выше неудовлетворённого — это ближе к смыслу ROC AUC.

`stack247` используется как начальный порядок (`base_margin`). Ranker учит только поправку, используя исходные признаки, digit-признаки и OOF-прогнозы 11 foundation-моделей как сигналы разногласия. Проверяем только фолд 0. Остальные фолды запускаем лишь при заметном положительном результате.

In [20]:
import xgboost as xgb

ranker_valid_fold = 0
ranker_train_mask = fold_numbers != ranker_valid_fold
ranker_valid_mask = fold_numbers == ranker_valid_fold

ranker_x_train = model_train.loc[
    ranker_train_mask, model_feature_columns
].copy()
ranker_x_valid = model_train.loc[
    ranker_valid_mask, model_feature_columns
].copy()
ranker_y_train = target[ranker_train_mask]
ranker_y_valid = target[ranker_valid_mask]

arrival_column = "Arrival Delay in Minutes"
ranker_arrival_median = ranker_x_train[arrival_column].median()
for frame in [ranker_x_train, ranker_x_valid]:
    frame[arrival_column] = frame[arrival_column].fillna(
        ranker_arrival_median
    )

class_values = sorted(model_train["Class"].dropna().unique())
class_mapping = {value: index for index, value in enumerate(class_values)}
for frame in [ranker_x_train, ranker_x_valid]:
    frame["Class"] = frame["Class"].map(class_mapping).astype(np.int8)
    add_digit_features(frame)

tfm_oof_paths = sorted((project_root / "data" / "tfm").glob("oof_*.npy"))
for tfm_oof_path in tfm_oof_paths:
    model_name = tfm_oof_path.stem.removeprefix("oof_")
    model_oof = np.load(tfm_oof_path).reshape(-1)
    assert len(model_oof) == len(target)
    ranker_x_train[f"tfm {model_name}"] = model_oof[ranker_train_mask]
    ranker_x_valid[f"tfm {model_name}"] = model_oof[ranker_valid_mask]

ranker_x_train["stack247 probability"] = stack247_oof[ranker_train_mask]
ranker_x_valid["stack247 probability"] = stack247_oof[ranker_valid_mask]

def safe_logit(probability):
    clipped = np.clip(np.asarray(probability), 1e-6, 1.0 - 1e-6)
    return np.log(clipped / (1.0 - clipped))

ranker_base_train = safe_logit(stack247_oof[ranker_train_mask])
ranker_base_valid = safe_logit(stack247_oof[ranker_valid_mask])

ranker_random = np.random.default_rng(20261031)
ranker_order = ranker_random.permutation(len(ranker_x_train))
ranker_group_size = 512
ranker_qid = np.arange(len(ranker_x_train)) // ranker_group_size

ranker_x_train_ordered = ranker_x_train.iloc[ranker_order].reset_index(drop=True)
ranker_y_train_ordered = ranker_y_train[ranker_order]
ranker_base_train_ordered = ranker_base_train[ranker_order]

print("train shape:", ranker_x_train_ordered.shape)
print("valid shape:", ranker_x_valid.shape)
print("ranking groups:", int(ranker_qid[-1] + 1))
print("foundation predictions:", len(tfm_oof_paths))

train shape: (559708, 50)
valid shape: (139927, 50)
ranking groups: 1094
foundation predictions: 11


In [21]:
pairwise_ranker = xgb.XGBRanker(
    objective="rank:pairwise",
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0.5,
    reg_lambda=10.0,
    tree_method="hist",
    lambdarank_pair_method="mean",
    lambdarank_num_pair_per_sample=8,
    random_state=20261031,
    n_jobs=4,
)

pairwise_ranker.fit(
    ranker_x_train_ordered,
    ranker_y_train_ordered,
    qid=ranker_qid,
    base_margin=ranker_base_train_ordered,
    verbose=True,
)

ranker_final_margin = pairwise_ranker.predict(
    ranker_x_valid,
    output_margin=True,
    base_margin=ranker_base_valid,
)
ranker_correction = ranker_final_margin - ranker_base_valid
ranker_base_auc = roc_auc_score(
    ranker_y_valid,
    stack247_oof[ranker_valid_mask],
)

ranker_alpha_grid = [0.0, 0.02, 0.05, 0.10, 0.20, 0.35, 0.50, 0.75, 1.0]
ranker_rows = []
for alpha in ranker_alpha_grid:
    corrected_margin = ranker_base_valid + alpha * ranker_correction
    corrected_auc = roc_auc_score(ranker_y_valid, corrected_margin)
    ranker_rows.append({
        "alpha": alpha,
        "auc": corrected_auc,
        "difference": corrected_auc - ranker_base_auc,
    })

ranker_results = pd.DataFrame(ranker_rows)
print(ranker_results.to_string(index=False, float_format=lambda value: f"{value:.9f}"))
print(f"\nbase fold 0 AUC: {ranker_base_auc:.9f}")
print("correction standard deviation:", float(ranker_correction.std()))

      alpha         auc   difference
0.000000000 0.962357578  0.000000000
0.020000000 0.962358584  0.000001006
0.050000000 0.962360232  0.000002654
0.100000000 0.962362559  0.000004981
0.200000000 0.962365096  0.000007518
0.350000000 0.962365775  0.000008196
0.500000000 0.962352325 -0.000005254
0.750000000 0.962236277 -0.000121301
1.000000000 0.961655840 -0.000701738

base fold 0 AUC: 0.962357578
correction standard deviation: 0.20330686537102943


Если лучший `difference` остаётся около нескольких миллионных или становится отрицательным, ranker отбрасываем. Для запуска остальных фолдов нужен заметный плюс, потому что выбор `alpha` на одном фолде сам по себе немного переобучается.

## Решающий повтор на фолде 1

На фолде 0 лучший результат составил примерно `+0.000008`. Это положительно, но ещё может быть случайностью. Перед четырьмя полными обучениями повторяем тот же неизменный алгоритм на фолде 1. Параметры и сетку `alpha` не подстраиваем.

In [22]:
for variable_name in [
    "pairwise_ranker",
    "ranker_x_train",
    "ranker_x_valid",
    "ranker_x_train_ordered",
    "ranker_y_train_ordered",
    "ranker_base_train_ordered",
]:
    globals().pop(variable_name, None)
gc.collect()

def run_pairwise_ranker_fold(valid_fold):
    train_mask = fold_numbers != valid_fold
    valid_mask = fold_numbers == valid_fold

    x_train = model_train.loc[train_mask, model_feature_columns].copy()
    x_valid = model_train.loc[valid_mask, model_feature_columns].copy()
    y_train = target[train_mask]
    y_valid = target[valid_mask]

    arrival_median = x_train["Arrival Delay in Minutes"].median()
    for frame in [x_train, x_valid]:
        frame["Arrival Delay in Minutes"] = (
            frame["Arrival Delay in Minutes"].fillna(arrival_median)
        )
        frame["Class"] = frame["Class"].map(class_mapping).astype(np.int8)
        add_digit_features(frame)

    for tfm_oof_path in tfm_oof_paths:
        model_name = tfm_oof_path.stem.removeprefix("oof_")
        model_oof = np.load(tfm_oof_path).reshape(-1)
        x_train[f"tfm {model_name}"] = model_oof[train_mask]
        x_valid[f"tfm {model_name}"] = model_oof[valid_mask]

    x_train["stack247 probability"] = stack247_oof[train_mask]
    x_valid["stack247 probability"] = stack247_oof[valid_mask]

    base_train = safe_logit(stack247_oof[train_mask])
    base_valid = safe_logit(stack247_oof[valid_mask])

    random_generator = np.random.default_rng(20261031 + valid_fold)
    order = random_generator.permutation(len(x_train))
    qid = np.arange(len(x_train)) // ranker_group_size

    model = xgb.XGBRanker(
        objective="rank:pairwise",
        n_estimators=500,
        learning_rate=0.03,
        max_depth=5,
        min_child_weight=20,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_alpha=0.5,
        reg_lambda=10.0,
        tree_method="hist",
        lambdarank_pair_method="mean",
        lambdarank_num_pair_per_sample=8,
        random_state=20261031,
        n_jobs=4,
    )
    model.fit(
        x_train.iloc[order].reset_index(drop=True),
        y_train[order],
        qid=qid,
        base_margin=base_train[order],
        verbose=True,
    )

    final_margin = model.predict(
        x_valid,
        output_margin=True,
        base_margin=base_valid,
    )
    correction = final_margin - base_valid
    base_auc = roc_auc_score(y_valid, stack247_oof[valid_mask])

    rows = []
    for alpha in ranker_alpha_grid:
        corrected_auc = roc_auc_score(
            y_valid,
            base_valid + alpha * correction,
        )
        rows.append({
            "fold": valid_fold,
            "alpha": alpha,
            "auc": corrected_auc,
            "difference": corrected_auc - base_auc,
        })

    del model, x_train, x_valid
    gc.collect()
    return pd.DataFrame(rows)

ranker_fold_one_results = run_pairwise_ranker_fold(valid_fold=1)
print(ranker_fold_one_results.to_string(
    index=False,
    float_format=lambda value: f"{value:.9f}",
))

 fold       alpha         auc   difference
    1 0.000000000 0.961410364  0.000000000
    1 0.020000000 0.961411053  0.000000689
    1 0.050000000 0.961412127  0.000001763
    1 0.100000000 0.961413462  0.000003098
    1 0.200000000 0.961414810  0.000004446
    1 0.350000000 0.961412169  0.000001805
    1 0.500000000 0.961395862 -0.000014502
    1 0.750000000 0.961273031 -0.000137332
    1 1.000000000 0.960600969 -0.000809395


Если фолд 1 тоже показывает положительный максимум примерно при том же `alpha`, направление можно проверить полностью. Если знак меняется или максимум снова находится на уровне нескольких миллионных, оставляем чистый V54 как финальный надёжный вариант.

## Рискованный шаг дальше V54

Это не CV-эксперимент, а одна контролируемая public-leaderboard попытка. В публичном Surface Atlas сохранены V41, V43 и V54. Два последних шага имеют похожее направление, а V54 стал лучшей версией. Продолжаем только самый свежий шаг `V43 → V54` ещё на 25% и затем снова превращаем значения в ранги.

Такой файл может дать небольшой плюс, остаться на том же уровне или ухудшиться. Он не является гарантированным способом получить 0.963 и не заменяет V54 как безопасную финальную отправку.

In [23]:
import zipfile

surface_atlas_path = project_root / "surface_atlas_full.zip"
with zipfile.ZipFile(surface_atlas_path) as surface_zip:
    v43_frame = pd.read_csv(
        surface_zip.open("rabari_v43_submission.csv")
    )
    v54_frame = pd.read_csv(
        surface_zip.open("rabari_v54_submission.csv")
    )

assert v43_frame["id"].equals(sample_submission["id"])
assert v54_frame["id"].equals(sample_submission["id"])

v43_rank = v43_frame["satisfaction"].rank(
    method="average", pct=True
).to_numpy()
v54_rank = v54_frame["satisfaction"].rank(
    method="average", pct=True
).to_numpy()

recent_direction = v54_rank - v43_rank
extrapolation_fraction = 0.25
extrapolated_score = (
    v54_rank + extrapolation_fraction * recent_direction
)
extrapolated_rank = pd.Series(extrapolated_score).rank(
    method="average", pct=True
).to_numpy()

v54_direction_submission = sample_submission.copy()
v54_direction_submission["satisfaction"] = extrapolated_rank
v54_direction_path = (
    project_root / "submissions" / "v54_recent_direction_plus25.csv"
)
v54_direction_submission.to_csv(v54_direction_path, index=False)

print("rows:", len(v54_direction_submission))
print("correlation with V54:", np.corrcoef(extrapolated_rank, v54_rank)[0, 1])
print("direction RMS:", float(recent_direction.std()))
print("applied step RMS:", float((extrapolated_rank - v54_rank).std()))
print("saved:", v54_direction_path)

rows: 299844
correlation with V54: 0.9999912246724413
direction RMS: 0.004840572586627015
applied step RMS: 0.0012093612335238712
saved: c:\Users\User\Desktop\Airline Satisfaction\submissions\v54_recent_direction_plus25.csv


Загружаем `v54_recent_direction_plus25.csv` как единственную рискованную проверку. Для финального выбора сохраняем чистый V54, пока новый файл не покажет результат выше.

## Последняя проверка направления: +100%

Шаг +25% сохранил public score `0.96209`, поэтому он оказался меньше разрешения leaderboard или лежит на плоском участке. Теперь делаем один полный дополнительный шаг той же длины, что `V43 → V54`. Это последняя проверка данного направления: при отсутствии улучшения дальше не экстраполируем.

In [24]:
full_step_score = v54_rank + recent_direction
full_step_rank = pd.Series(full_step_score).rank(
    method="average", pct=True
).to_numpy()

v54_full_step_submission = sample_submission.copy()
v54_full_step_submission["satisfaction"] = full_step_rank
v54_full_step_path = (
    project_root / "submissions" / "v54_recent_direction_plus100.csv"
)
v54_full_step_submission.to_csv(v54_full_step_path, index=False)

print("correlation with V54:", np.corrcoef(full_step_rank, v54_rank)[0, 1])
print("applied step RMS:", float((full_step_rank - v54_rank).std()))
print("saved:", v54_full_step_path)

correlation with V54: 0.9998600884402321
applied step RMS: 0.004828932935282792
saved: c:\Users\User\Desktop\Airline Satisfaction\submissions\v54_recent_direction_plus100.csv


Если `v54_recent_direction_plus100.csv` не превышает `0.96209`, чистый V54 остаётся финальным вариантом, а дальнейшие шаги по этому направлению считаем leaderboard-переобучением.